# Arquitectura medallón con PySpark: ejecución paso a paso

**Objetivo.** Construir las capas Bronze, Silver y Gold con un conjunto de datos de ventas que contiene errores intencionales, y comprobar que el resultado coincide con la simulación HTML.

**Requisitos.**
- Python con `pyspark` instalado (`pip install pyspark`) y un JDK compatible con su versión de Spark.
- Estructura de carpetas: este notebook y, junto a él, la carpeta `datos/` con `ventas_marzo.csv`.

**Instrucciones.** Ejecute las celdas en orden con *Shift + Enter*. Lea el comentario de cada paso antes de ejecutarlo.

## Paso 0. Preparar el entorno

Se crea una sesión de Spark en **modo local** (`local[*]`: un hilo por núcleo lógico). La variable `GUARDAR_EN_DISCO` controla si cada capa se escribe en Parquet. Déjela en `False` si trabaja en Windows sin `winutils` configurado; cámbiela a `True` para persistir las capas en la carpeta `lakehouse/`.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable          # mismo Python de Anaconda para el driver y los workers
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Medallon_Ventas")
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

RUTA_CSV = "datos/ventas_marzo.csv"
GUARDAR_EN_DISCO = False

def guardar(df, capa):
    if GUARDAR_EN_DISCO:
        df.write.mode("overwrite").parquet(f"lakehouse/{capa}/ventas")

def leer(capa, df_memoria):
    return spark.read.parquet(f"lakehouse/{capa}/ventas") if GUARDAR_EN_DISCO else df_memoria

print("Versión de Spark:", spark.version, "| hilos disponibles:", spark.sparkContext.defaultParallelism)

## Paso 1. Capa Bronze: datos crudos

Se leen los datos **sin corregir nada** y todo como texto (`inferSchema` desactivado), para no perder información ante formatos inesperados. Se agregan metadatos de ingesta: número de fila, archivo de origen y fecha de ingesta.

In [ ]:
bronze = (spark.read.option("header", True).option("inferSchema", False).csv(RUTA_CSV)
          .withColumn("_fila", F.monotonically_increasing_id())
          .withColumn("_archivo", F.lit("ventas_marzo.csv"))
          .withColumn("_ingesta", F.current_timestamp()))
guardar(bronze, "bronze")

print("Filas en Bronze:", bronze.count())
bronze.orderBy("_fila").show(truncate=False)
bronze.printSchema()

## Paso 2. Diagnosticar la calidad de Bronze

Antes de depurar, se miden los problemas. Observe que el monto `S/ 900` es válido tras limpiar el símbolo, mientras que `abc` no lo es. La función `try_cast` devuelve nulo ante un valor inválido; con `cast` simple, Spark 4 (modo ANSI) lanzaría una excepción.

In [ ]:
print("Filas:", bronze.count(), "| ids distintos:", bronze.select("id").distinct().count())
print("Clientes vacíos:", bronze.filter(F.col("cliente").isNull()).count())
print("Montos vacíos:", bronze.filter(F.col("monto").isNull()).count())

print("Montos con texto no convertible:")
(bronze.withColumn("monto_txt", F.regexp_replace("monto", r"S/|\s", ""))
       .withColumn("monto_num", F.expr("try_cast(monto_txt as double)"))
       .filter(F.col("monto").isNotNull() & F.col("monto_num").isNull())
       .select("id", "monto").show())

print("Regiones tal como llegaron:")
bronze.select(F.concat(F.lit("["), "region", F.lit("]")).alias("region")).distinct().show()

## Paso 3. Capa Silver: depuración y validación

Se define una función con las reglas de calidad, para poder reutilizarla y cambiar reglas más adelante:

- **R1.** Eliminar duplicados por `id`.
- **R2.** Estandarizar la región (quitar espacios y unificar mayúsculas).
- **R3.** Convertir el monto a número.
- **R4.** Rechazar filas sin cliente o sin monto.

Las filas rechazadas no se descartan en silencio: pasan a una tabla de **cuarentena** con el motivo.

In [ ]:
def construir_silver(bronze_df, quitar_duplicados=True, estandarizar_region=True):
    w = Window.partitionBy("id").orderBy("_fila")
    ev = (bronze_df
          .withColumn("rn", F.row_number().over(w))
          .withColumn("monto_txt", F.regexp_replace("monto", r"S/|\s", ""))
          .withColumn("monto_num", F.expr("try_cast(monto_txt as double)"))
          .withColumn("region_final", F.initcap(F.trim("region")) if estandarizar_region else F.col("region")))
    motivo = (F.when(F.lit(quitar_duplicados) & (F.col("rn") > 1), "Duplicado (mismo id)")
               .when(F.col("cliente").isNull() | F.col("monto").isNull(), "Cliente o monto vacío")
               .when(F.col("monto_num").isNull(), "Monto no convertible a número"))
    ev = ev.withColumn("motivo", motivo)
    silver = (ev.filter(F.col("motivo").isNull())
                .select("id", "fecha", "cliente", F.col("region_final").alias("region"),
                        "producto", F.col("monto_num").alias("monto")))
    cuarentena = (ev.filter(F.col("motivo").isNotNull())
                    .select("id", "fecha", "cliente", "region", "producto", "monto", "motivo"))
    return silver, cuarentena

bronze_leido = leer("bronze", bronze)
silver, cuarentena = construir_silver(bronze_leido)
guardar(silver, "silver")

print("Silver:", silver.count(), "filas | Cuarentena:", cuarentena.count(), "filas")
silver.orderBy(F.col("id").cast("int")).show(truncate=False)
print("Filas rechazadas:")
cuarentena.orderBy(F.col("id").cast("int")).show(truncate=False)

## Paso 4. Capa Gold: agregación para el negocio

Se resume Silver por región: ventas totales, número de ventas y ticket promedio. Esta tabla es la que consumiría un reporte gerencial.

In [ ]:
def construir_gold(silver_df):
    return (silver_df.groupBy("region")
            .agg(F.round(F.sum("monto"), 2).alias("ventas"),
                 F.count("*").alias("n_ventas"),
                 F.round(F.avg("monto"), 2).alias("ticket_promedio"))
            .orderBy(F.col("ventas").desc()))

gold = construir_gold(leer("silver", silver))
guardar(gold, "gold")
gold.show(truncate=False)

## Paso 5. Verificar contra la simulación

Los resultados deben coincidir con los del simulador HTML: 12 filas en Bronze, 7 en Silver, 5 en cuarentena y un total de ventas de 6345,5. Si alguna comprobación falla, revise la ruta del CSV y su contenido.

In [ ]:
assert bronze.count() == 12
assert silver.count() == 7
assert cuarentena.count() == 5
total = gold.agg(F.sum("ventas")).first()[0]
assert abs(total - 6345.5) < 1e-9
print("Verificación superada. Total de ventas:", total)

## Paso 6. Experimento: ¿qué pasa si falla una regla?

Se reconstruye Silver y Gold **desde el mismo Bronze**, sin volver a ingestar. Compare con el resultado correcto del Paso 4 y responda las preguntas de reflexión.

In [ ]:
print("Sin estandarizar la región (R2 desactivada):")
s_sin_region, _ = construir_silver(bronze_leido, estandarizar_region=False)
construir_gold(s_sin_region).show(truncate=False)

print("Sin eliminar duplicados (R1 desactivada):")
s_sin_dup, _ = construir_silver(bronze_leido, quitar_duplicados=False)
construir_gold(s_sin_dup).show(truncate=False)
print("Total de ventas inflado:", construir_gold(s_sin_dup).agg(F.sum("ventas")).first()[0])

## Paso 7. Particiones y tareas paralelas (opcional)

Relacione esta sesión con los simuladores anteriores. Cada partición se procesa como una tarea, y las tareas se reparten entre los hilos de `local[*]`. Mientras la sesión esté activa, la interfaz web de Spark suele estar disponible en `http://localhost:4040` (el puerto puede variar si está ocupado).

In [ ]:
print("Particiones de Bronze:", bronze.rdd.getNumPartitions())
rep = bronze.repartition(4)
print("Particiones tras repartition(4):", rep.rdd.getNumPartitions())
print("Filas por partición:", rep.rdd.glom().map(len).collect())
print("Interfaz web:", spark.sparkContext.uiWebUrl)

## Preguntas de reflexión

1. ¿Por qué Bronze conserva los errores en lugar de corregirlos al ingresar?
2. En el Paso 6, ¿qué ocurrió con las regiones al desactivar R2? ¿Qué decisión errónea podría tomar un gerente?
3. ¿Cuánto se infló el total sin R1? ¿Quién detectaría ese error si solo existiera Gold?
4. Si cambia una regla de negocio, ¿qué capas se recalculan y cuál permanece intacta?
5. ¿Qué ventajas y qué costos tendría persistir cada capa en disco (Parquet o Delta)?

In [ ]:
spark.stop()